```mermaid
flowchart LR
    A0["00"] --> A1a["01a"] --> A1b["01b"] --> A2["02"] --> A3["03"] --> A4a["04a"] --> A4b["04b"]
    A4b --> A5a["05a"] --> A5b["05b"] --> A6a["06a"] --> A6b["06b"]
    A6b --> A7["07"] --> A8a["08a"] --> A8b["08b"]
    A8b --> A9["09"] --> A10["10"] --> A11["11"] --> A12["12"] 
    
    classDef normal fill:#f8f9fa,stroke:#adb5bd,stroke-width:1px,color:#111;
    classDef done fill:#e8f7f0,stroke:#198754,stroke-width:1.5px,color:#111;
    classDef current fill:#fff3cd,stroke:#ff8c00,stroke-width:2px,color:#111;
    
    class A0,A1a,A1b,A2,A3,A4a,A4b,A5a,A5b,A6a,A6b,A7 done;
    class A8a current;
    class A8b,A9,A10,A11,A12 normal;
```

# Notebook 08 — Embeddings: Similarity Geometry, Clustering/Search, and Model Inputs

This notebook introduces **dense** vector representations called *embeddings* as a complementary representation to Bag-of-Words and TF–IDF features.

In previous notebooks, we explored **sparse** lexical representations based on counts and weighted word overlap. Embeddings instead represent text as points in a high-dimensional semantic space, where proximity reflects patterns of usage and contextual similarity.

---

**Sparse representations** (like one-hot encoding or bag-of-words/TF-IDF vectors) have a dimension for every word in the vocabulary. If the vocabulary has 50,000 words, each vector has 50,000 dimensions — but almost all of those entries are zero. A one-hot vector for "philosophy" has a single 1 and 49,999 zeros. Even a TF-IDF vector for a whole document is mostly zeros, since most documents only use a tiny fraction of the vocabulary.

---

**Dense representations** (embeddings) pack the meaning of a word or text into a much smaller vector — typically 100 to 1,000 dimensions — where almost every entry is a non-zero, meaningful number. Nothing is wasted on "this word is absent." Instead, each dimension contributes some (often uninterpretable) piece of information, and meaning is encoded in the overall pattern of values across all dimensions rather than in which single dimension is "switched on."

---

Dense representations increase:

- efficiency, because a 300-dimensional dense vector can represent a word about as usefully as a 50,000-dimensional sparse one, because information is distributed across dimensions rather than one-per-word.

- the possibilities to analyse semantic similarity because dense vectors place semantically related words near each other in the vector space (via cosine similarity or Euclidean distance), you get relationships like "reason" and "rationality" being close together, or the famous king − man + woman ≈ queen pattern. Sparse one-hot vectors can't do this at all: every word is equally (maximally) distant from every other word by construction, since exactly one dimension differs.

The central methodological question of this notebook is:

> How does representing text geometrically change what kinds of similarity and historical relationships become visible?

We focus on four goals:

- generating reusable document/chunk embeddings
- exploring semantic neighborhoods through nearest-neighbor search
- visualizing embedding geometry and temporal structure
- preparing reusable embedding matrices for later modeling tasks

**Important methodological point:** embedding dimensions are not directly interpretable. We interpret embeddings through relationships: similarity, clustering, neighborhoods, trajectories, and representative examples.

## Learning goals

By the end of this notebook, students should be able to:

- explain the difference between lexical and embedding representations
- generate sentence/document embeddings with a pretrained model
- perform semantic nearest-neighbor search
- interpret cosine similarity geometrically
- visualize embedding spaces with dimensionality reduction
- compare time bins using centroid embeddings
- critically evaluate embedding-based similarity claims
- cache and reuse embeddings for downstream modeling

## Method note

Unlike TF–IDF, embeddings are *dense* representations learned from large corpora. Modern embedding models capture statistical regularities in contextual usage patterns, which often allows semantically related passages to appear close together even when they do not share many exact words.

However, embeddings also introduce methodological challenges:

- embedding dimensions are not human-interpretable
- similarity can reflect stylistic or corpus artifacts
- nearest-neighbor results may appear convincing even when they are unstable
- visualization methods compress very high-dimensional spaces into only 2 dimensions

For this reason, this notebook emphasizes *evaluation-by-inspection*: we inspect retrieved passages, compare representations, and interpret embedding outputs cautiously rather than treating them as ground truth.

In [ ]:
from __future__ import annotations

from pathlib import Path
import re
import json
from collections import Counter

from tqdm.auto import tqdm

import numpy as np
import pandas as pd

from scipy import sparse

from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors

import matplotlib.pyplot as plt
import seaborn as sns

import spacy
from spacy.tokens import DocBin

# Optional plotting backend for interactive figures
try:
    import plotly.express as px
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except Exception:
    HAS_PLOTLY = False

# Sentence-transformers
try:
    from sentence_transformers import SentenceTransformer
    HAS_ST = True
except Exception:
    HAS_ST = False
    print('sentence-transformers not available.')
    print('Install sentence-transformers')

In [ ]:
# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
PROJECT_ROOT = Path('.')

DATA_DIR = PROJECT_ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'

TEXTS_DIR = PROCESSED_DIR / 'gutenberg' / 'texts_cleaned'

ANALYSIS_DIR = PROJECT_ROOT / 'analysis'
FIGURES_DIR = ANALYSIS_DIR / 'figures'
TABLES_DIR = ANALYSIS_DIR / 'tables'
REPORTS_DIR = ANALYSIS_DIR / 'reports'
MODELS_DIR = ANALYSIS_DIR / 'models'

CACHE_DIR = PROJECT_ROOT / 'cache'

for p in [FIGURES_DIR, TABLES_DIR, REPORTS_DIR, MODELS_DIR, CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Canonical document index from earlier notebooks
DOC_INDEX = TABLES_DIR / 'nb03-doc_index.csv'

# Split DocBin files from Notebook 05a
SPLIT_DIR = PROCESSED_DIR / 'nb05-corpus-split'

# Embedding cache paths
CHUNKS_PATH = TABLES_DIR / 'nb08-chunks.parquet'
EMB_PATH = CACHE_DIR / 'nb08-chunk_embeddings.npy'
CHUNK_INDEX_PATH = TABLES_DIR / 'nb08-chunks_index.csv'

# Embedding configuration
EMBEDDING_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
BATCH_SIZE = 32

# Chunking controls
MIN_CHUNK_CHARS = 400
MAX_CHUNK_CHARS = 2000

print('DOC_INDEX:', DOC_INDEX)
print('SPLIT_DIR:', SPLIT_DIR)
print('Embedding model:', EMBEDDING_MODEL_NAME)

## Load the canonical document table

We reuse the canonical document index generated earlier in the workflow. This keeps document identifiers, filenames, and time bins stable across notebooks.

In [ ]:
df = pd.read_csv(DOC_INDEX)
df["publication_year"] = pd.to_numeric(df["publication_year"], errors="coerce").astype("Int64")

print('Documents:', len(df))
display(df.head())

## Load the split spaCy corpus from Notebook 05a

Notebook 05a serialized the corpus into split `DocBin` files. This allows us to reload annotated documents without rerunning linguistic annotation.

Why does this matter?
- annotation is computationally expensive
- serialized artifacts improve reproducibility
- later notebooks can focus on analysis rather than preprocessing

# Fill the gap

In [ ]:
spacy_files = sorted(SPLIT_DIR.glob('*.spacy'))
print(f'Found {len(spacy_files)} split files.')

# =============================================== YOUR CODE HERE ===============================================
nlp =                                # Load spacy's lightweight pipeline vocabulary and disable the NER pipeline

docs = []
for fp in spacy_files:
    db = DocBin().from_disk(fp)
    docs.extend(list(db.get_docs(nlp.vocab)))

print('Loaded docs:', len(docs))

## Reconstruct document texts

Embeddings require raw text strings. We reconstruct them from the spaCy documents and align them with the canonical document table.

In [ ]:
chunk_rows = []
for doc in docs:
    # Retrieve metadata from the Doc
    pg_id = doc.user_data.get('pg_id')
    title = doc.user_data.get('title') 
    pub_year = doc.user_data.get('publication_year') 
    time_bin = doc.user_data.get('time_bin')
    chunk_index = doc.user_data.get('chunk_index', 0)
    
    chunk_rows.append({
        'pg_id': pg_id,
        'title': title,
        'publication_year': pub_year,
        'time_bin': time_bin,
        'chunk_index': chunk_index,
        'text': doc.text,
        'n_tokens': len(doc)
    })

chunk_df = pd.DataFrame(chunk_rows)
chunk_df["publication_year"] = pd.to_numeric(chunk_df["publication_year"], errors="coerce").astype("Int64")
chunk_df.to_parquet(CHUNKS_PATH, index=False)
print(f"\nCreated chunk DataFrame with {len(chunk_df)} rows.")
display(chunk_df.head())

# Chunking documents for embeddings

Embedding models usually work best on relatively short spans of text. Entire philosophical books are often too long and internally heterogeneous.

We therefore split books into paragraph-sized chunks. This gives us:
- more manageable semantic units
- better retrieval quality
- finer-grained temporal and conceptual exploration

**Reflection question:** how might chunk size affect the kinds of similarity we observe?

In [ ]:
def chunk_paragraphs(text: str) -> list[str]:
    """
    Split a text into paragraph-sized chunks and trim extreme lengths.

    We use paragraphs because they often correspond to coherent argumentative units
    in philosophical writing.
    """

    paras = [p.strip() for p in re.split(r'\n\s*\n+', str(text)) if p.strip()]

    chunks = []
    for p in paras:
        if len(p) < MIN_CHUNK_CHARS:
            continue

        if len(p) > MAX_CHUNK_CHARS:
            p = p[:MAX_CHUNK_CHARS]

        chunks.append(p)

    return chunks

# Example
example_chunks = chunk_paragraphs(chunk_df.iloc[0]['text'])
print('\nExample chunks:', len(example_chunks), '\n')
print(example_chunks[0][:500])

# Build or load cached chunks + embeddings

Embeddings can take time to compute on CPU. We therefore cache:

- the chunk table
- the embedding matrix

so later notebooks can load them instantly without recomputation.

## Why normalize embeddings?

Every embedding is a vector, that is a list of numbers, and it has a length, called the L2 norm. The "2" refers to how it is calculated: you square each number in the vector, add the squares together, and take the square root, following the same principle as the Pythagorean theorem, though usually extended from two dimensions to hundreds. A vector's L2 norm in NLP can be large or small depending on factors like how often a word appeared during training, which has nothing to do with its meaning.


L2-normalizing a vector means rescaling it so its L2 norm becomes exactly 1, while keeping the direction it points in unchanged. Think of it like shrinking or stretching an arrow until it's exactly one unit long, without rotating it. Once every embedding has been L2-normalized this way, all vectors are directly comparable on equal terms — differences in length can no longer distort a similarity comparison, since every vector has the same length by construction. Only direction is left to compare, and direction is where an embedding's meaning is encoded: two words with similar meanings point in similar directions.


This is also why L2-normalizing pays off computationally. Comparing two vectors' directions (their cosine similarity) normally requires dividing by both vectors' lengths as part of the calculation. But once every vector already has length 1, that division becomes unnecessary — the raw dot product (a fast, simple calculation: multiply matching numbers together and sum them) gives the same answer directly. That's the shortcut behind the sentence: L2-normalization is what allows cosine similarity to be computed as a plain dot product.

This matters because:
- cosine similarity measures *directional* similarity
- normalized vectors avoid length effects
- nearest-neighbor search becomes more stable

**Key intuition:** embeddings that point in similar directions represent semantically similar passages.

### BATCH_SIZE trade-off (transformer encoding): 
Unlike the earlier rule-based EntityRuler pipeline, this one runs an actual neural network per batch, so
batch size has a much bigger effect on speed here — especially on GPU.
- CPU only: keep this small (8-32); larger batches mainly just use more RAM
  without much speedup, since there's no parallel hardware to exploit.
- GPU available: push this up (64-256) to better utilize the GPU; the limit
  is VRAM, not CPU cores. If you hit a CUDA out-of-memory error, halve it.
- Longer chunks (more tokens per chunk) need a smaller batch size than short
  ones for the same memory budget, since memory scales with tokens x batch.
When in doubt, start at 16-32 and watch GPU memory usage (nvidia-smi) or
just time a few batch sizes and compare. There is rarely a need to tune
this precisely.

# Fill the gap

In [ ]:
# Embedding configuration
EMBEDDING_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
# =============================================== YOUR CODE HERE ===============================================
BATCH_SIZE =              # Choose the right BATCH_SIZE

In [ ]:
from math import ceil

if EMB_PATH.exists():
    print('Loading cached embeddings.')
    E = np.load(EMB_PATH)

    if len(E) != len(chunk_df):
        raise ValueError(
            f"Mismatch: chunk_df has {len(chunk_df)} rows but embeddings have {len(E)} vectors"
        )

else:
    print('\nLoading embedding model...')
    model = SentenceTransformer(EMBEDDING_MODEL_NAME)
    print('Model loaded.')

    texts = chunk_df['text'].astype(str).tolist()
    emb_batches = []
    n_batches = ceil(len(texts) / BATCH_SIZE)

    print(f'\nEncoding {len(texts):,} chunks in {n_batches} batches...')

    for start in tqdm(
        range(0, len(texts), BATCH_SIZE),
        total=n_batches,
        desc='Encoding chunks',
        unit='batch'
    ):
        batch = texts[start:start + BATCH_SIZE]
        emb = model.encode(
            batch,
            batch_size=BATCH_SIZE,
            show_progress_bar=False,
            normalize_embeddings=True,
            convert_to_numpy=True,
        )
        emb_batches.append(emb)

    E = np.vstack(emb_batches)

    if len(E) != len(chunk_df):
        raise ValueError(
            f"Post-encode mismatch: chunk_df has {len(chunk_df)} rows but embeddings have {len(E)} vectors"
        )

    np.save(EMB_PATH, E)
    print('Saved embeddings:', EMB_PATH)

# Save lightweight chunk index (without raw text)
chunk_df.drop(columns=['text'], errors='ignore').to_csv(Path('./analysis/tables/nb08-chunk_emb_index.csv'), index=True)
chunk_df.drop(columns=['text'], errors='ignore').to_parquet(Path('./analysis/tables/nb08-chunk_emb_index.parquet'), index=True)
print('\nSaved chunk index to', Path('./analysis/tables/nb08-chunk_emb_index.csv'))
print('Saved chunk index parquet to', Path('./analysis/tables/nb08-chunk_emb_index.parquet'))

# Save embeddings in a parquet file where row 0 corresponds to chunk_df.index[0]
embedding_df = pd.DataFrame(E, index=chunk_df.index)
embedding_df.to_parquet(Path('./analysis/tables/nb08-emb_index.parquet'))
print('\nSaved indexed embedding parquet to', Path('./analysis/tables/nb08-emb_index.parquet'))

print('\nEmbedding matrix shape:', E.shape)
print('Embedding dtype:', E.dtype)

# Semantic nearest-neighbor search

One of the most intuitive embedding applications is semantic search:

- represent the query as an embedding
- compare it to chunk embeddings
- retrieve the nearest neighbors

Unlike TF–IDF, embeddings can retrieve semantically related passages even when they do not share many exact words.

In [ ]:
# Build nearest-neighbor index
nn = NearestNeighbors(n_neighbors=25, metric='cosine').fit(E)

def snippet(text: str, n: int = 240) -> str:
    text = re.sub(r'\s+', ' ', str(text)).strip()
    return text[:n] + ('…' if len(text) > n else '')


def query_neighbors(query: str, k: int = 10):
    """
    Retrieve the top-k semantic neighbors for a query string.

    Returns:
    - cosine similarity
    - metadata
    - short text snippet
    """

    q = model.encode([query], normalize_embeddings=True)

    dists, idx = nn.kneighbors(q, n_neighbors=k)

    out = chunk_df.iloc[idx[0]].copy()
    out['cosine_sim'] = 1 - dists[0]
    out['snippet'] = out['text'].map(lambda t: snippet(t, n=240))

    cols = [
        'cosine_sim',
        'pg_id',
        'title',
        'publication_year',
        'time_bin',
        'chunk_index',
        'snippet',
    ]

    return out[cols]

## Evaluation-by-inspection

Embedding outputs can appear convincing even when they are misleading.

We therefore inspect:
- retrieved passages
- time bins
- stylistic similarities
- possible retrieval artifacts

**Reflection questions:**
- Are the retrieved chunks semantically similar, or merely stylistically similar?
- Do embeddings retrieve paraphrases better than TF–IDF?
- Which kinds of concepts seem easiest or hardest to retrieve?

In [ ]:
queries = [
    'reason and experience',
    'freedom and justice',
    'nature and virtue',
    'knowledge and truth',
]

for q in queries:
    print('\nQuery:', q)
    display(query_neighbors(q, k=12))

# Visualizing embedding geometry

Embeddings live in high-dimensional spaces (often hundreds of dimensions). To visualize them, we compress them into 2 dimensions using PCA.

Important caveat:
- dimensionality reduction is a projection
- some relationships are lost
- distances in 2D are only approximations of relationships in the full space

We use visualization as an exploratory aid, not as proof. Unlike TF–IDF, which we explored through **Truncated SVD**, embeddings are already **dense numerical vectors**. This means they can be projected into two dimensions with a general dimensionality-reduction method such as **Principal Component Analysis (PCA)**.

The goal of this plot is not to interpret the individual dimensions of the embeddings directly. Instead, PCA gives us a simplified 2D view of the broader geometric structure of the embedding space. Chunks that appear close together in the plot are relatively similar under the embedding representation, while more distant chunks are less similar.

This kind of visualization is useful for exploring whether chunks cluster by period, theme, or other metadata. At the same time, it should be interpreted cautiously: PCA compresses a high-dimensional space into only two dimensions, so it provides an informative overview rather than a complete picture of semantic structure.

In [ ]:
# Sample chunks for visualization (avoid plotting tens of thousands of points)
MAX_PLOT = 1500

plot_idx = np.arange(len(chunk_df))
if len(plot_idx) > MAX_PLOT:
    rng = np.random.default_rng(42)
    plot_idx = rng.choice(plot_idx, size=MAX_PLOT, replace=False)

# PCA projection
pca = PCA(n_components=2, random_state=42)
Z = pca.fit_transform(E[plot_idx])

plot_df = chunk_df.iloc[plot_idx].copy()
plot_df['pc1'] = Z[:, 0]
plot_df['pc2'] = Z[:, 1]

# ---- Determine chronological order of time_bins ----
# Extract unique bins (drop NaN)
unique_bins = plot_df['time_bin'].dropna().unique()

def get_start_year(bin_str):
    """Extract the first year from a time_bin string like '1679–1860'."""
    try:
        return int(bin_str.split('–')[0])
    except (ValueError, AttributeError, IndexError):
        return 0   # fallback

# Sort bins by start year
sorted_bins = sorted(unique_bins, key=get_start_year)

plt.figure(figsize=(10, 7))

sns.scatterplot(
    data=plot_df,
    x='pc1',
    y='pc2',
    hue='time_bin',
    hue_order=sorted_bins, 
    alpha=0.6,
    s=30,
)

plt.title('Chunk embeddings projected to 2D (PCA)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend(title='Time bin', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

print('Explained variance ratio:', round(float(pca.explained_variance_ratio_.sum()), 3))

In [ ]:
# ---- Configuration ----
MAX_PLOT = 1500

# ---- Sample a subset of chunks ----
plot_idx = np.arange(len(chunk_df))
if len(plot_idx) > MAX_PLOT:
    rng = np.random.default_rng(42)
    plot_idx = rng.choice(plot_idx, size=MAX_PLOT, replace=False)

# ---- PCA projection (E is your chunk embedding matrix) ----
pca = PCA(n_components=2, random_state=42)
Z = pca.fit_transform(E[plot_idx])

plot_df = chunk_df.iloc[plot_idx].copy()
plot_df['pc1'] = Z[:, 0]
plot_df['pc2'] = Z[:, 1]

# ---- Ensure publication year is numeric (use column 'pub_year') ----
plot_df['year'] = pd.to_numeric(plot_df['publication_year'], errors='coerce')

# ---- Create figure and axis ----
fig, ax = plt.subplots(figsize=(10, 7))

# ---- Scatter plot with continuous color gradient ----
scatter = ax.scatter(
    x=plot_df['pc1'],
    y=plot_df['pc2'],
    c=plot_df['year'],          # numeric column for colour
    cmap='crest',               # colormap
    alpha=0.6,
    s=30
)

# ---- Add colorbar ----
cbar = fig.colorbar(scatter, ax=ax)
cbar.set_label('Publication Year')

# ---- Labels and title ----
ax.set_title('Chunk embeddings projected to 2D (PCA) – coloured by publication year')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')

plt.tight_layout()
plt.show()

print('Explained variance ratio:', round(float(pca.explained_variance_ratio_.sum()), 3))

In [ ]:
# ---- Configuration ----
MAX_PLOT = 1500

# ---- Sample a subset of chunks ----
plot_idx = np.arange(len(chunk_df))
if len(plot_idx) > MAX_PLOT:
    rng = np.random.default_rng(42)
    plot_idx = rng.choice(plot_idx, size=MAX_PLOT, replace=False)

# ---- PCA projection ----
pca = PCA(n_components=2, random_state=42)
Z = pca.fit_transform(E[plot_idx])

plot_df = chunk_df.iloc[plot_idx].copy()
plot_df['pc1'] = Z[:, 0]
plot_df['pc2'] = Z[:, 1]

# ---- Ensure publication year is numeric ----
plot_df['year'] = pd.to_numeric(plot_df['publication_year'], errors='coerce')

# ---- Add a short text snippet for hover ----
plot_df['snippet'] = plot_df['text'].str[:150] + '…'

# ---- Create interactive scatter plot ----
fig = px.scatter(
    plot_df,
    x='pc1',
    y='pc2',
    color='year',                     # continuous colour scale
    color_continuous_scale='mint', # same as matplotlib's viridis
    hover_data={
        'title': True,
        'time_bin': True,
        'year': True,
        'snippet': True,
        'pc1': False,   # hide from hover to reduce clutter
        'pc2': False
    },
    title='Chunk embeddings projected to 2D (PCA) – coloured by publication year',
    labels={'pc1': 'PC1', 'pc2': 'PC2', 'year': 'Publication year'},
    opacity=0.6,
    height=700,
)

# ---- Improve layout ----
fig.update_traces(marker=dict(size=6))
fig.update_layout(
    hoverlabel=dict(bgcolor="white", font_size=12),
    legend_title_text='Year'
)

fig.show(renderer="notebook")

print('Explained variance ratio:', round(float(pca.explained_variance_ratio_.sum()), 3))

## Interpreting the PCA Plot

The first two principal components (PC1 and PC2) capture the largest sources of variance in the chunk embeddings. In this corpus, PC1 is strongly correlated with **publication year** – it represents the historical shift from early (left) to late (right) texts. PC2 captures a secondary, weaker pattern (e.g., genre, author style, or topic).

| Position | What it means |
| :--- | :--- |
| **Close to the x‑axis (PC1)** | The chunk is well‑explained by the dominant historical trend. Its vocabulary aligns with the typical language of its period. It is not unusual for its time. |
| **Far from the x‑axis (high \|PC2\|)** | The chunk contains additional variation *beyond* the historical shift. It may be unusual in genre, style, or topic compared to other texts from the same period. These are the **outsiders** or **atypical texts**. |
| **Close to the y‑axis (PC2)** | The chunk has a near‑zero score on the secondary component. It is "average" with respect to the secondary pattern – it does not stand out in terms of genre or style beyond the temporal shift. |
| **Far from the origin (both high \|PC1\| and high \|PC2\|)** | These chunks are **extreme** in both dimensions. They are the most distinctive texts in the corpus – either very early and stylistically unusual, or very late and stylistically unusual. Hover over them (in the Plotly version) to see their titles and snippets. |

# Time-bin centroid embeddings

To compare historical periods, we compute one centroid embedding per time bin by averaging chunk embeddings within that bin.

This gives a compact representation of each period in semantic space.

We can then compare periods using cosine similarity.

In [ ]:
# Keep only chunks with a valid time bin
m = chunk_df['time_bin'].notna().to_numpy()
chunk_df2 = chunk_df.loc[m].copy()
E2 = E[m]

# Group indices per bin
groups = chunk_df2.groupby('time_bin').indices

def _bin_sort_key(x):
    s = str(x).replace('–', '-').replace('−', '-')
    m = re.search(r'-?\d+', s)
    return int(m.group(0)) if m else 10**9

labels = sorted(groups.keys(), key=_bin_sort_key)

centroids = []
counts = []

for b in labels:
    idx = np.array(list(groups[b]), dtype=int)

    Cb = E2[idx].mean(axis=0)
    Cb = Cb / (np.linalg.norm(Cb) + 1e-12)

    centroids.append(Cb)
    counts.append(len(idx))

C = np.vstack(centroids)

# Cosine similarity between bins
S = cosine_similarity(C)

sim_df = pd.DataFrame(S, index=[str(b) for b in labels], columns=[str(b) for b in labels])

display(sim_df.round(3))

## Interpreting similarity heatmaps

Each cell in the heatmap compares two *time bins*:
- values near 1 indicate highly similar centroid embeddings
- lower values indicate larger semantic differences

Because embedding similarities often cluster near 1.0, we tighten the heatmap scale to make differences visible.

In [ ]:
S_off = S.copy()
np.fill_diagonal(S_off, np.nan)

vmin = float(np.nanpercentile(S_off, 5))

plt.figure(figsize=(10, 8))

sns.heatmap(
    sim_df,
    cmap='crest',
    vmin=vmin,
    vmax=1.0,
)

plt.title('Embedding similarity across time bins (centroid cosine)')
plt.xlabel('Time bin')
plt.ylabel('Time bin')
plt.tight_layout()
plt.show()

print('Off-diagonal similarity range:')
print('Min:', round(float(np.nanmin(S_off)), 4))
print('Median:', round(float(np.nanmedian(S_off)), 4))
print('Max:', round(float(np.nanmax(S_off)), 4))

# Semantic drift across time bins

We now compute the cosine distance between *consecutive* time bins.

Interpretation:
- small distance = periods remain semantically similar
- larger distance = stronger semantic drift

Important caveat: this is a *corpus-level* shift measure. Changes may reflect:
- conceptual change
- changing authors/subfields
- sampling differences
- uneven corpus composition

We therefore interpret these curves cautiously.

In [ ]:
cos_sim_next = np.sum(C[1:] * C[:-1], axis=1)
cos_dist_next = 1 - cos_sim_next

shift = pd.DataFrame({
    'time_bin': labels[1:],
    'cosine_distance_to_prev': cos_dist_next,
})

plt.figure(figsize=(10, 4))

plt.plot(
    shift['time_bin'],
    shift['cosine_distance_to_prev'],
    marker='o',
)

plt.xticks(rotation=45, ha='right')
plt.title('Semantic shift by time bin (embedding centroid distance)')
plt.ylabel('Cosine distance to previous bin')
plt.xlabel('Time bin')
plt.tight_layout()
plt.show()

display(shift.sort_values('cosine_distance_to_prev', ascending=False).head(10))

# Representative chunks for a time bin

Embedding centroids summarize many chunks. To interpret them, we retrieve the chunks closest to a chosen centroid.

This helps connect abstract vector geometry back to actual textual evidence.

In [ ]:
def representative_chunks(bin_label, k: int = 5):
    idx = np.array(list(groups[bin_label]), dtype=int)

    cent = C[labels.index(bin_label)]

    sims = cosine_similarity(E2[idx], cent.reshape(1, -1)).ravel()

    top = idx[np.argsort(-sims)[:k]]

    out = chunk_df.loc[top, [
        'pg_id',
        'title',
        'publication_year',
        'time_bin',
        'chunk_index',
        'text',
    ]].copy()

    out['centroid_cosine'] = np.sort(sims)[::-1][:k]

    return out

example_bin = labels[0]
print('Example time bin:', example_bin)
display(representative_chunks(example_bin, k=5))

In [ ]:
from sklearn.manifold import TSNE
# from umap import UMAP 

# t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
Z_tsne = tsne.fit_transform(E[plot_idx])

# Or UMAP (faster for large data)
# umap = UMAP(n_components=2, random_state=42)
# Z_umap = umap.fit_transform(E[plot_idx])

plot_df['tsne1'] = Z_tsne[:, 0]
plot_df['tsne2'] = Z_tsne[:, 1]

# ---- Determine chronological order of time_bins ----
# Extract unique bins (drop NaN)
unique_bins = plot_df['time_bin'].dropna().unique()

# Sort bins by start year
sorted_bins = sorted(unique_bins, key=get_start_year)

sns.scatterplot(data=plot_df, x='tsne1', y='tsne2', hue='time_bin', hue_order=sorted_bins, alpha=0.6, s=20)
plt.title('Chunk embeddings – t‑SNE projection')
plt.legend(bbox_to_anchor=(1.02, 1))
plt.show()

### Close Reading of Temporal Transitions

This cell identifies concrete text chunks that illustrate the semantic shifts
between time periods. It uses three complementary views:
    
    1. Transitional chunks – texts closest to the midpoint between two consecutive centroids.
    
    2. Bridge chunks – texts from a period that are more similar to the next period.
    
    3. Outlier chunks – texts from other periods that are closest to a given centroid.

In [ ]:
# ------------------------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------------------------
def get_closest_chunks(centroid, top_k=10, exclude_pg_ids=None):
    """
    Return chunks with highest cosine similarity to a given centroid.
    """
    sims = cosine_similarity(E, centroid.reshape(1, -1)).ravel()
    nn = np.argsort(-sims)[:top_k]
    rows = chunk_df.iloc[nn].copy()
    rows['similarity'] = sims[nn]
    return rows[['pg_id', 'title', 'time_bin', 'chunk_index', 'similarity', 'text']]

def show_transition(centroid_a, centroid_b, label_a, label_b, top_k=5):
    """
    Show chunks that are closest to the *midpoint* between two centroids.
    These are the texts that are most "transitional" between periods.
    """
    midpoint = (centroid_a + centroid_b) / 2
    midpoint = midpoint / np.linalg.norm(midpoint)
    
    sims = cosine_similarity(E, midpoint.reshape(1, -1)).ravel()
    nn = np.argsort(-sims)[:top_k]
    rows = chunk_df.iloc[nn].copy()
    rows['similarity_to_midpoint'] = sims[nn]
    
    print(f"\nTransitional chunks between {label_a} and {label_b}:")
    display(rows[['title', 'time_bin', 'similarity_to_midpoint', 'text']])

def find_bridge_chunks(centroids, labels, threshold=0.05):
    """
    For each time bin, find chunks that are more similar to the *next* period
    than to their own (i.e., they anticipate future language).
    """
    bridge_list = []
    
    for i in range(len(centroids) - 1):
        own_centroid = centroids[i]
        next_centroid = centroids[i+1]
        own_label = labels[i]
        next_label = labels[i+1]
        
        # Filter chunks from this period
        mask = chunk_df['time_bin'] == own_label
        if mask.sum() == 0:
            continue
        E_sub = E[mask]
        
        sim_own = cosine_similarity(E_sub, own_centroid.reshape(1, -1)).ravel()
        sim_next = cosine_similarity(E_sub, next_centroid.reshape(1, -1)).ravel()
        
        # Find chunks where similarity to next > similarity to own + threshold
        bridge_mask = sim_next > (sim_own + threshold)
        if bridge_mask.sum() > 0:
            bridge_indices = np.where(mask)[0][bridge_mask]
            for idx, sim in zip(bridge_indices, sim_next[bridge_mask]):
                bridge_list.append({
                    'pg_id': chunk_df.iloc[idx]['pg_id'],
                    'title': chunk_df.iloc[idx]['title'],
                    'own_period': own_label,
                    'closer_to': next_label,
                    'similarity_to_next': sim,
                    'text': chunk_df.iloc[idx]['text'][:200] + '…'
                })
    
    return pd.DataFrame(bridge_list)

def show_outlier_chunks(centroid, label, top_k=5):
    """
    Show chunks from other periods that are closest to a given centroid.
    """
    sims = cosine_similarity(E, centroid.reshape(1, -1)).ravel()
    # Exclude chunks from the same period
    mask = chunk_df['time_bin'] != label
    if mask.sum() == 0:
        print(f"No chunks outside {label} to show.")
        return
    sims_masked = sims.copy()
    sims_masked[~mask] = -1   # ignore same-period chunks
    nn = np.argsort(-sims_masked)[:top_k]
    rows = chunk_df.iloc[nn].copy()
    rows['similarity_to_centroid'] = sims[nn]
    print(f"\nChunks from other periods closest to {label} centroid:")
    display(rows[['title', 'time_bin', 'similarity_to_centroid', 'text']])

In [ ]:
# ------------------------------------------------------------------------------
# Compute time_bins and centroids automatically (self-contained)
# ------------------------------------------------------------------------------
# 1. Get unique time bins, drop NaN, and sort chronologically
unique_bins = chunk_df['time_bin'].dropna().unique()

time_bins = sorted(unique_bins, key=get_start_year)

print(f"Found {len(time_bins)} time periods: {time_bins}")

# 2. Compute centroid for each time bin (mean of chunk embeddings)
centroids = []
for tb in time_bins:
    mask = chunk_df['time_bin'] == tb
    if mask.sum() > 0:
        centroid = E[mask].mean(axis=0)
        # Normalise to unit vector (for cosine similarity)
        centroid = centroid / (np.linalg.norm(centroid) + 1e-12)
        centroids.append(centroid)
    else:
        centroids.append(np.zeros(E.shape[1]))

# Convert to numpy array for easier handling
centroids = np.array(centroids)

print(f"Computed centroids for {len(centroids)} periods.")

# ------------------------------------------------------------------------------
# Run analyses
# ------------------------------------------------------------------------------

# 1. Show transitional chunks for each consecutive pair
print("\n" + "="*60)
print("TRANSITIONAL CHUNKS (between consecutive time periods)")
print("="*60)

for i in range(len(time_bins) - 1):
    show_transition(
        centroids[i],
        centroids[i+1],
        time_bins[i],
        time_bins[i+1],
        top_k=3
    )

# 2. Find and display bridge chunks
print("\n" + "="*60)
print("BRIDGE CHUNKS (anticipating future periods)")
print("="*60)

bridge_df = find_bridge_chunks(centroids, time_bins, threshold=0.05)
if len(bridge_df) > 0:
    print(f"\nFound {len(bridge_df)} bridge chunks:")
    display(bridge_df[['title', 'own_period', 'closer_to', 'similarity_to_next', 'text']].head(20))
else:
    print("No bridge chunks found with the current threshold.")

# 3. Show outlier chunks for the first and last period (as examples)
print("\n" + "="*60)
print("OUTLIER CHUNKS (other periods closest to a centroid)")
print("="*60)

if len(time_bins) >= 2:
    show_outlier_chunks(centroids[0], time_bins[0], top_k=5)
    show_outlier_chunks(centroids[-1], time_bins[-1], top_k=5)

print("\nAnalysis complete. Use the displayed snippets for close reading.")

### Binary Classification: Early vs Late Periods

In this exercise, we train a simple logistic regression model to predict
whether a chunk comes from the *earliest* or the *latest* time period.
This tests whether the vocabulary differences we observed in the keyness
analysis are strong enough to be *predictive*.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# ------------------------------------------------------------------------------
# 1. Prepare the binary labels
# ------------------------------------------------------------------------------
# Get earliest and latest time bins (chronologically sorted)
unique_bins = chunk_df['time_bin'].dropna().unique()

time_bins = sorted(unique_bins, key=get_start_year)
early_bin = time_bins[0]
late_bin = time_bins[-1]

print(f"\nEarly bin: {early_bin}")
print(f"Late bin: {late_bin}")

# Filter chunks belonging to these two bins
mask = (chunk_df['time_bin'] == early_bin) | (chunk_df['time_bin'] == late_bin)
binary_df = chunk_df[mask].copy()
binary_df['label'] = np.where(binary_df['time_bin'] == late_bin, 1, 0)

print(f"\nTotal chunks in early period: {(binary_df['label'] == 0).sum()}")
print(f"Total chunks in late period: {(binary_df['label'] == 1).sum()}")

In [ ]:
# ------------------------------------------------------------------------------
# 2. Balance the dataset (optional but recommended)
# ------------------------------------------------------------------------------
# To avoid the model simply learning the class imbalance, we take an equal
# number of chunks from each period (the size of the smaller class).
n_early = (binary_df['label'] == 0).sum()
n_late = (binary_df['label'] == 1).sum()
sample_size = min(n_early, n_late)

balanced_df = pd.concat([
    binary_df[binary_df['label'] == 0].sample(n=sample_size, random_state=42),
    binary_df[binary_df['label'] == 1].sample(n=sample_size, random_state=42)
]).sample(frac=1, random_state=42)  # shuffle

print(f"\nBalanced dataset: {len(balanced_df)} chunks ({sample_size} per class)")

In [ ]:
# ------------------------------------------------------------------------------
# 3. Vectorize the text (TF‑IDF)
# ------------------------------------------------------------------------------
from sklearn.feature_extraction.text import TfidfVectorizer


vectorizer = TfidfVectorizer(
    max_features=5000,        # keep only top 5000 terms for speed
    stop_words='english',
    ngram_range=(1, 2),       # unigrams + bigrams
    min_df=3
)

X = vectorizer.fit_transform(tqdm(balanced_df['text'], desc="Vectorizing", total=len(balanced_df['text'])))
y = balanced_df['label'].values

print(f"\nTF‑IDF matrix shape: {X.shape}")

In [ ]:
# ------------------------------------------------------------------------------
# 4. Train / test split and logistic regression
# ------------------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, y_train)

# ------------------------------------------------------------------------------
# 5. Evaluation
# ------------------------------------------------------------------------------
y_pred = clf.predict(X_test)

print("\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)
print(classification_report(y_test, y_pred, target_names=[early_bin, late_bin]))

print("\n" + "="*60)
print("CONFUSION MATRIX")
print("="*60)
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[early_bin, late_bin])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# ------------------------------------------------------------------------------
# 6. Top predictive words (coefficients)
# ------------------------------------------------------------------------------
# Get feature names
feature_names = vectorizer.get_feature_names_out()
coefficients = clf.coef_[0]  # shape (n_features,)

# Top words pushing toward LATE period (positive coefficients)
top_late_indices = np.argsort(coefficients)[-15:][::-1]
print("\nTop 15 words characteristic of late period:")
for idx in top_late_indices:
    print(f"  {feature_names[idx]}: {coefficients[idx]:.4f}")

# Top words pushing toward EARLY period (negative coefficients)
top_early_indices = np.argsort(coefficients)[:15]
print("\nTop 15 words characteristic of early period:")
for idx in top_early_indices:
    print(f"  {feature_names[idx]}: {coefficients[idx]:.4f}")

In [ ]:
# ------------------------------------------------------------------------------
# 7. Manual test: predict a few random chunks
# ------------------------------------------------------------------------------
print("\n" + "="*60)
print("MANUAL TEST: Predict a few random chunks")
print("="*60)

test_chunks = chunk_df.sample(5, random_state=42)
test_texts = test_chunks['text'].tolist()
test_labels = test_chunks['time_bin'].tolist()
X_test_manual = vectorizer.transform(test_texts)
predictions = clf.predict(X_test_manual)

for i, (text, true_label, pred) in enumerate(zip(test_texts, test_labels, predictions)):
    pred_label = late_bin if pred == 1 else early_bin
    print(f"\nChunk {i+1}:")
    print(f"  True period: {true_label}")
    print(f"  Predicted: {pred_label}")
    print(f"  Snippet: {text[:150]}...")

# Reflection questions

- How do embedding-based neighbors differ from TF–IDF neighbors?
- Which time bins appear most semantically similar? Why might that be?
- Which historical transitions show the strongest semantic drift?
- Do the retrieved chunks support the interpretation suggested by the geometry?
- Which limitations of embeddings become visible through manual inspection?

# Summary

In this notebook we:

- generated reusable chunk embeddings
- explored semantic nearest-neighbor retrieval
- visualized embedding geometry
- compared historical periods through centroid embeddings
- measured semantic drift across time bins
- connected embedding geometry back to textual evidence

These embedding representations will later serve as:
- exploratory tools
- clustering/search representations
- features for downstream supervised models
- possible inputs for semantic-shift analysis

```mermaid
flowchart TB
    A0["00<br/>Bootcamp"] --> P1

    subgraph P1["Part I — Corpus building and analysis"]
        direction LR
        A1a["01a<br/>Corpus metadata"] --> A1b["01b<br/>Corpus building"] --> A2["02<br/>Preprocessing"] --> A3["03<br/>Distributions + time"] --> A4a["04a<br/>Lexical exploration"] --> A4b["04b<br/>Embedding"]
    end

    subgraph P2["Part II — Linguistic annotations"]
        direction LR
        A5a["05a<br/>spaCy annotation"] --> A5b["05b<br/>Relation extraction"] --> A6a["06a<br/>NER"] --> A6b["06b<br/>Custom NER"]
    end

    subgraph P3["Part III — Representations"]
        direction LR
        A7["07<br/>BoW + TF-IDF"] --> A8a["08a<br/>Embeddings"] --> A8b["08b<br/>Transformers"]
    end

    subgraph P4["Part IV — Models and interpretation"]
        direction LR
        A9["09<br/>Classification"] --> A10["10<br/>Custom NER training"] --> A11["11<br/>Topic modeling"] --> A12["12<br/>Semantic shift"]
    end

    P1 --> P2
    P2 --> P3
    P3 --> P4

    classDef start fill:#f3f0ff,stroke:#6f42c1,stroke-width:1.5px,color:#111;
    classDef prep fill:#eef7ff,stroke:#1f77b4,stroke-width:1.5px,color:#111;
    classDef annot fill:#eefaf0,stroke:#2ca02c,stroke-width:1.5px,color:#111;
    classDef repr fill:#fff7e6,stroke:#ff8c00,stroke-width:1.5px,color:#111;
    classDef model fill:#fff0f0,stroke:#d62728,stroke-width:1.5px,color:#111;

    classDef highlight fill:#fff3b0,stroke:#f5a623,stroke-width:4px,color:#111;

    class A1a,A1b,A2,A3,A4a,A4b prep;
    class A5a,A5b,A6a,A6b annot;
    class A7,A8a,A8b repr;
    class A9,A10,A11,A12 model;

    class A8a highlight;
```